# Flujo de trabajo de clasificación: Predicción de diabetes

## ¿Qué es la clasificación?

La **clasificación** es un tipo de machine learning supervisado que se usa para predecir **etiquetas categóricas** (clases discretas). A diferencia de la regresión (que predice valores continuos), la clasificación asigna observaciones a categorías predefinidas.

**Ejemplos:**
- Detección de spam en correos electrónicos (spam vs. no spam)
- Reconocimiento de imágenes (gato vs. perro)
- **Nuestro caso**: Diagnóstico de diabetes (tiene diabetes vs. no tiene diabetes)

## Regresión vs. Clasificación: El mismo dominio, problemas distintos

Este notebook usa datos de diabetes, igual que el notebook de regresión, pero resuelve un problema diferente:

| Aspecto | Notebook de Regresión | Este Notebook de Clasificación |
|--------|---------------------|------------------------------|
| **Pregunta** | ¿Cuánto progresará la enfermedad? | ¿El paciente tiene diabetes? |
| **Tipo de target** | Continuo (25-346) | Binario (0 o 1) |
| **Tipo de problema** | Regresión | Clasificación |

## Lo que aprenderemos

Este notebook demuestra el **mismo flujo de trabajo de ML** que la regresión, pero para clasificación:

1. **Cargar y explorar datos** - Entender las features y las clases target
2. **Dividir los datos** - Separar training y test
3. **Entrenar modelos** - Ajustar diferentes algoritmos de clasificación
4. **Evaluar el rendimiento** - Comparar modelos con métricas de clasificación
5. **Visualizar resultados** - Entender predicciones con confusion matrices

## El dataset

Usaremos la **base de datos de diabetes de los Indios Pima**:
- **768 pacientes femeninas** de herencia india Pima
- **8 features**: embarazos, glucosa, presión arterial, grosor de piel, insulina, IMC, función de pedigrí, edad
- **Target**: Resultado binario (0 = sin diabetes, 1 = tiene diabetes)

## 1. Importar librerías

In [ ]:
# Manipulación de datos y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Descarga de datos
import kagglehub

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report,
    ConfusionMatrixDisplay
)

# Establecer semilla aleatoria para reproducibilidad
np.random.seed(42)

# Mejorar la apariencia de los gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Cargar y explorar los datos

### Descargar el dataset

In [ ]:
# Descargar la última versión de Kaggle
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
print("Ruta a los archivos del dataset:", path)

### Cargar los datos

In [ ]:
# Cargar el archivo CSV
import os
csv_path = os.path.join(path, 'diabetes.csv')
df = pd.read_csv(csv_path)

print(f"Forma del dataset: {df.shape}")
print(f"Número de muestras: {df.shape[0]}")
print(f"Número de features: {df.shape[1] - 1}")
print(f"\nNombres de columnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()

### Descripción de las features

- **Pregnancies**: Número de veces embarazada
- **Glucose**: Concentración de glucosa en plasma (2 horas en prueba de tolerancia oral a la glucosa)
- **BloodPressure**: Presión arterial diastólica (mm Hg)
- **SkinThickness**: Grosor del pliegue cutáneo del tríceps (mm)
- **Insulin**: Insulina sérica de 2 horas (mu U/ml)
- **BMI**: Índice de masa corporal (peso en kg/(estatura en m)²)
- **DiabetesPedigreeFunction**: Función de pedigrí de diabetes (influencia genética)
- **Age**: Edad en años
- **Outcome**: Variable de clase (0 = sin diabetes, 1 = tiene diabetes)

### Información del dataset

In [ ]:
df.info()

### Estadísticas básicas

In [ ]:
df.describe().round(2)

### Verificar el tipo de la variable target

La variable target es **binaria** (0 o 1), confirmando que este es un **problema de clasificación**.

In [ ]:
print(f"Valores únicos de la variable target (Outcome): {df['Outcome'].unique()}")
print(f"Tipo de dato de la variable target: {df['Outcome'].dtype}")
print(f"\nDistribución de clases:")
print(df['Outcome'].value_counts().sort_index())
print(f"\nBalance de clases:")
print(df['Outcome'].value_counts(normalize=True).sort_index())

### Visualizar la distribución de clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
class_counts = df['Outcome'].value_counts().sort_index()
axes[0].bar(['Sin Diabetes', 'Con Diabetes'], class_counts.values, 
            color=['#66c2a5', '#fc8d62'], edgecolor='black')
axes[0].set_ylabel('Conteo')
axes[0].set_title('Distribución de Clases')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Gráfico de pastel
axes[1].pie(class_counts.values, labels=['Sin Diabetes', 'Con Diabetes'], 
            autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'], startangle=90)
axes[1].set_title('Proporciones de Clases')

plt.tight_layout()
plt.show()

print("El target es binario (0/1) -> Este es un problema de CLASIFICACIÓN")

## 3. Preprocesamiento de datos

### Manejo de valores faltantes

En este dataset, un valor de **0** en features como Glucose, BloodPressure, SkinThickness, Insulin y BMI indica datos faltantes (es físicamente imposible tener presión arterial o IMC de 0).

Lo manejaremos así:
1. Reemplazando `0` con `NaN` en estas columnas específicas.
2. Imputando (rellenando) los valores `NaN` con la **mediana** de la columna. La mediana es más robusta frente a outliers que la media.

In [ ]:
# Features donde el 0 indica datos faltantes
features_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Reemplazar 0 con NaN
df[features_with_zeros] = df[features_with_zeros].replace(0, np.nan)

# Verificar cuántos valores faltantes tenemos ahora
print("Valores faltantes (NaN) tras reemplazar los 0s:")
print(df[features_with_zeros].isnull().sum())

# Imputar con la mediana
for feature in features_with_zeros:
    median_val = df[feature].median()
    df[feature].fillna(median_val, inplace=True)
    print(f"Rellenado {feature} con mediana: {median_val:.1f}")

print("\nValores faltantes manejados!")

### Distribuciones de features por clase

In [ ]:
# Seleccionar algunas features clave para visualizar
key_features = ['Glucose', 'BMI', 'Age', 'Pregnancies']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, feature in enumerate(key_features):
    # Separar los datos por clase
    no_diabetes = df[df['Outcome'] == 0][feature]
    has_diabetes = df[df['Outcome'] == 1][feature]
    
    axes[idx].hist([no_diabetes, has_diabetes], bins=20, label=['Sin Diabetes', 'Con Diabetes'],
                   color=['#66c2a5', '#fc8d62'], alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].set_title(f'Distribución de {feature} por Clase')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Análisis de correlaciones

In [ ]:
# Calcular correlaciones con el target
correlations = df.corr()['Outcome'].drop('Outcome').sort_values(ascending=False)

# Graficar
plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in correlations]
correlations.plot(kind='barh', color=colors, edgecolor='black')
plt.xlabel('Correlación con Diabetes')
plt.title('Correlaciones de Features con el Target')
plt.axvline(0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nTop 3 features más correlacionadas con diabetes:")
print(correlations.head(3))

## 3. Dividir los datos: Training vs. Test

### ¿Por qué dividir los datos?

Mismo principio que en regresión:
- **Training set (80%)**: Enseñar al modelo
- **Test set (20%)**: Evaluar con datos no vistos

**Adicional para clasificación**: Usamos `stratify` para mantener el balance de clases en ambos conjuntos.

### Preparar features y target

In [ ]:
# Separar features (X) y target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print(f"Forma de las features: {X.shape}")
print(f"Forma del target: {y.shape}")
print(f"\nNombres de features: {list(X.columns)}")

### Realizar la división (con estratificación)

In [ ]:
# División con estratificación para mantener el balance de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para test
    random_state=42,    # Para reproducibilidad
    stratify=y          # Mantener proporciones de clases
)

print("Resumen de la división de datos:")
print("=" * 50)
print(f"Total de muestras: {len(X)}")
print(f"Muestras de training: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Muestras de test: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

print(f"\nBalance de clases en el training set:")
print(y_train.value_counts(normalize=True).sort_index())
print(f"\nBalance de clases en el test set:")
print(y_test.value_counts(normalize=True).sort_index())
print("\nProporciones de clases mantenidas!")

### Visualizar la división

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Proporciones de la división
sizes = [len(X_train), len(X_test)]
labels = [f'Train\n{len(X_train)} muestras', f'Test\n{len(X_test)} muestras']
colors = ['#66c2a5', '#fc8d62']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
axes[0].set_title('División Train/Test')

# Comparación de distribución de clases
train_counts = y_train.value_counts().sort_index()
test_counts = y_test.value_counts().sort_index()
x = np.arange(2)
width = 0.35
axes[1].bar(x - width/2, train_counts.values, width, label='Train', edgecolor='black')
axes[1].bar(x + width/2, test_counts.values, width, label='Test', edgecolor='black')
axes[1].set_ylabel('Conteo')
axes[1].set_title('Distribución de Clases: Train vs. Test')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Sin Diabetes', 'Con Diabetes'])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Escalado de features

### ¿Por qué escalar?

La Logistic Regression (y muchos algoritmos) funciona mejor con features escaladas porque:
- Las features tienen rangos diferentes (ej., Age: 20-80, BMI: 0-67)
- El escalado asegura que todas las features contribuyan por igual
- Mejora la velocidad de convergencia

**StandardScaler** transforma las features para tener media=0 y desviación estándar=1.

In [ ]:
# Crear y ajustar el scaler con los datos de training
scaler = StandardScaler()
scaler.fit(X_train)

# Transformar ambos conjuntos
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Escalado de features completado!")
print(f"\nAntes del escalado:")
print(f"  Los rangos de features varían (ej., Age: {X_train['Age'].min():.0f}-{X_train['Age'].max():.0f}, "
      f"Glucose: {X_train['Glucose'].min():.0f}-{X_train['Glucose'].max():.0f})")
print(f"\nDespués del escalado:")
print(f"  Todas las features tienen media ≈ 0, desv. estándar ≈ 1")
print(f"  Media: {X_train_scaled.mean():.6f}")
print(f"  Desv. estándar: {X_train_scaled.std():.6f}")

## 5. Entrenar los modelos

Entrenaremos dos modelos de clasificación:

1. **Logistic Regression**: Clasificador lineal (análogo a la Linear Regression)
2. **Decision Tree Classifier**: Clasificador no lineal (análogo al Decision Tree Regressor)

### Modelo 1: Logistic Regression

In [ ]:
# Crear y entrenar el modelo de Logistic Regression
model_logistic = LogisticRegression(max_iter=1000, random_state=42)
model_logistic.fit(X_train_scaled, y_train)

# Hacer predicciones en el test set
y_pred_logistic = model_logistic.predict(X_test_scaled)

### Modelo 2: Decision Tree Classifier

In [ ]:
# Crear y entrenar el modelo de Decision Tree (no requiere escalado, pero usamos el escalado para comparación justa)
model_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
model_tree.fit(X_train_scaled, y_train)

# Hacer predicciones en el test set
y_pred_tree = model_tree.predict(X_test_scaled)
print(f"Profundidad del árbol: {model_tree.get_depth()}")
print(f"Número de hojas: {model_tree.get_n_leaves()}")

## 6. Evaluar los modelos

### Métricas de clasificación

¡Distintas de la regresión! Usamos:

1. **Accuracy**: Porcentaje de predicciones correctas
   - Rango: 0 a 1 (más alto es mejor)
   - Puede ser engañosa con clases desbalanceadas

2. **Precision**: De los positivos predichos, ¿cuántos son realmente positivos?
   - Fórmula: TP / (TP + FP)

3. **Recall (Sensibilidad)**: De los positivos reales, ¿cuántos detectamos?
   - Fórmula: TP / (TP + FN)

4. **F1-Score**: Media armónica de Precision y Recall
   - Equilibra Precision y Recall

In [ ]:
# Calcular accuracy para ambos modelos
acc_logistic = accuracy_score(y_test, y_pred_logistic)
acc_tree = accuracy_score(y_test, y_pred_tree)

print("Comparación de rendimiento de modelos")
print("=" * 60)
print(f"{'Modelo':<25} {'Accuracy':>15}")
print("-" * 60)
print(f"{'Logistic Regression':<25} {acc_logistic:>15.3f}")
print(f"{'Decision Tree':<25} {acc_tree:>15.3f}")
print("=" * 60)

# Determinar el ganador
if acc_logistic > acc_tree:
    print("\nLogistic Regression tiene mejor rendimiento")
else:
    print("\nDecision Tree tiene mejor rendimiento")

### Informes de clasificación detallados

In [ ]:
print("Logistic Regression - Informe de clasificación:")
print("=" * 60)
print(classification_report(y_test, y_pred_logistic, 
                          target_names=['Sin Diabetes', 'Con Diabetes']))

print("\nDecision Tree - Informe de clasificación:")
print("=" * 60)
print(classification_report(y_test, y_pred_tree,
                          target_names=['Sin Diabetes', 'Con Diabetes']))

### Crear DataFrame de comparación

In [ ]:
# Extraer métricas de los informes de clasificación
from sklearn.metrics import precision_score, recall_score, f1_score

results_df = pd.DataFrame({
    'Modelo': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [acc_logistic, acc_tree],
    'Precision': [
        precision_score(y_test, y_pred_logistic, average='weighted'),
        precision_score(y_test, y_pred_tree, average='weighted')
    ],
    'Recall': [
        recall_score(y_test, y_pred_logistic, average='weighted'),
        recall_score(y_test, y_pred_tree, average='weighted')
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_logistic, average='weighted'),
        f1_score(y_test, y_pred_tree, average='weighted')
    ]
})

results_df = results_df.round(3)
results_df

## 7. Visualizar los resultados

### Confusion matrices

Una confusion matrix muestra:
- **True Positives (TP)**: Diabetes predicha correctamente
- **True Negatives (TN)**: Sin diabetes predicha correctamente
- **False Positives (FP)**: Predicción de diabetes, pero el paciente no la tiene
- **False Negatives (FN)**: Predicción de no diabetes, pero el paciente sí la tiene

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix de Logistic Regression
cm_logistic = confusion_matrix(y_test, y_pred_logistic)
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_logistic, 
                                display_labels=['Sin Diabetes', 'Con Diabetes'])
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title(f'Logistic Regression\nAccuracy = {acc_logistic:.3f}')

# Confusion matrix de Decision Tree
cm_tree = confusion_matrix(y_test, y_pred_tree)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_tree,
                                display_labels=['Sin Diabetes', 'Con Diabetes'])
disp2.plot(ax=axes[1], cmap='Oranges', values_format='d')
axes[1].set_title(f'Decision Tree\nAccuracy = {acc_tree:.3f}')

plt.tight_layout()
plt.show()

### Importancia de features (Decision Tree)

In [ ]:
# Obtener importancias de features
importances = model_tree.feature_importances_
feature_names = X.columns

# Crear DataFrame y ordenar
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Graficar
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], edgecolor='black')
plt.xlabel('Importancia')
plt.title('Importancia de Features (Decision Tree)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 3 features más importantes:")
print(importance_df.head(3).to_string(index=False))

## Resumen

### Lo que aprendimos

1. **La clasificación predice categorías** (vs. la regresión que predice valores continuos)
2. **Mismo dominio, problema distinto**: Ambos notebooks usan datos de diabetes
   - Regresión: Predecir la progresión de la enfermedad (¿cuánto?)
   - Clasificación: Predecir la presencia de diabetes (¿sí o no?)
3. **El flujo de trabajo es similar**:
   - Cargar datos → Dividir → Escalar → Entrenar → Evaluar
4. **Métricas diferentes**:
   - Regresión: MAE, RMSE, R²
   - Clasificación: Accuracy, Precision, Recall, F1
5. **Visualizaciones diferentes**:
   - Regresión: Diagramas de dispersión, residuos
   - Clasificación: Confusion matrix
6. **La estratificación importa** en clasificación para mantener el balance de clases

### Comparación con el notebook de regresión

| Aspecto | Regresión | Clasificación |
|--------|-----------|----------------|
| **Dataset** | UCI Diabetes | Indios Pima |
| **Target** | Progresión de la enfermedad (continuo) | Tiene diabetes (binario) |
| **Pregunta** | ¿Cuánto? | ¿Cuál clase? |
| **Modelos** | LinearRegression, DecisionTreeRegressor | LogisticRegression, DecisionTreeClassifier |
| **Métricas** | MAE, RMSE, R² | Accuracy, Precision, Recall, F1 |
| **Visualizaciones** | Diagramas de dispersión, residuos | Confusion matrix |
| **Escalado** | No requerido para modelos de árbol | Importante para LogisticRegression |

### Conclusión clave

El **flujo de trabajo de ML es el mismo**, pero:
- El **tipo de problema** determina si usar regresión o clasificación
- Las **métricas y visualizaciones** varían según el tipo de variable target